# Week 6 - Bayesian Optimization

Generate optimized recommendations for Week 6 using utility modules.

## Setup

In [ ]:
import numpy as np
import warnings
import sys
import importlib
sys.path.append('..')  # Add parent directory to path

# Import utility modules (reload to pick up any code changes)
import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import propose_next_point, fit_gp, get_strategy

from utils.data_utils import (
    load_week_data,
    save_week_data,
    combine_with_week_results, 
    print_data_summary
)

## 1. Load Week 5 Data

Load the combined data from previous weeks

In [ ]:
# Load Week 5 clean data
inputs, outputs = load_week_data("../week 5/week5_clean_data.npz")
print_data_summary(inputs, outputs, "Week 5 Data")

## 2. Add Week 5 Results

In [ ]:
# Week 5 submitted points
week5_inputs = {
    1: np.array([0.413000, 0.410000]),
    2: np.array([0.696140, 0.120362]),
    3: np.array([0.413540, 0.169341, 0.828542]),
    4: np.array([0.413690, 0.367443, 0.365391, 0.413441]),
    5: np.array([1.000000, 1.000000, 0.980000, 1.000000]),
    6: np.array([0.905469, 0.120580, 0.787406, 0.739169, 0.134727]),
    7: np.array([0.000000, 0.409516, 0.783624, 0.238064, 0.409213, 0.773381]),
    8: np.array([0.027475, 0.220302, 0.000000, 0.107071, 1.000000, 0.105289, 0.280000, 0.999469])
}

# Week 5 outputs (received from black box)
week5_outputs = {
    1: 0.6113945125903011,
    2: 0.5471132047651114,
    3: -0.07005706411221177,
    4: 0.7101451761837265,
    5: 8290.379382930587,
    6: -0.9899044720268426,
    7: 1.4719848360079488,
    8: 9.7274406567879
}

# Combine with Week 5 results
inputs, outputs = combine_with_week_results(inputs, outputs, week5_inputs, week5_outputs)
print_data_summary(inputs, outputs, "After Week 5 Results")

In [ ]:
# Save combined data for Week 7
save_week_data(inputs, outputs, "week6_clean_data.npz")

## 3. Week 5 Results Analysis

Evaluate which strategies worked and which failed to inform Week 6 approach.

In [ ]:
# Week 5 results analysis
print("=" * 70)
print("WEEK 5 RESULTS ANALYSIS")
print("=" * 70)

# Best values before Week 5 query
best_before_w5 = {}
for fid in range(1, 9):
    best_before_w5[fid] = np.max(outputs[fid][:-1])

print(f"\n{'F':>2} {'Dims':>4} {'Best Before W5':>14} {'W5 Query':>14} {'New Best':>14} {'Status'}")
print("-" * 70)

improved = 0
for fid in range(1, 9):
    dim = inputs[fid].shape[1]
    prev_best = best_before_w5[fid]
    w5_val = week5_outputs[fid]
    new_best = np.max(outputs[fid])
    
    if w5_val >= prev_best:
        status = "NEW BEST"
        improved += 1
    else:
        status = f"miss (best still {prev_best:.4f})"
    
    print(f"{fid:>2} {dim:>3}D {prev_best:>14.4f} {w5_val:>14.4f} {new_best:>14.4f}   {status}")

print(f"\nWeek 5 hit rate: {improved}/8 functions improved")
print("=" * 70)

## 4. Sensitivity Analysis

Updated sensitivity analysis with Week 5 data to inform Week 6 strategies.

In [ ]:
from utils.sensitivity import sensitivity_analysis

for func_id in range(1, 9):
    sensitivity_analysis(func_id, inputs[func_id], outputs[func_id])

## 5. Week 6 Strategy Design

Strategies based on 5-week performance history, updated sensitivity analysis, and GP kernel diagnostics.

### Week 5 lessons learned:
- **Manual nudges beat optimizers on narrow spikes** — F1's dim2+0.005 gave +47% improvement
- **Submitted queries that diverged from plan performed worst** — F3, F6 both regressed
- **Don't move hyper-sensitive dimensions** — F6 dims 4&5 (ls=0.001, 0.0003) must be locked
- **F5's corner optimum is confirmed** — dim3 pull to 0.98 lost 372 points
- **EI with tight bounds remains the most reliable** — F4 improved for 4th consecutive week

### Updated kernel length scales (with W5 data):

| F | Length scales | Key finding |
|---|-------------|-------------|
| F1 | [0.012, 0.008] | Ultra-narrow spike — GP unreliable, manual nudges work |
| F2 | [0.025, 1.015] | Only dim1 matters (dim2 now irrelevant with more data) |
| F3 | [0.004, 0.015, 0.004] | All dims hyper-sensitive — stay within 1 length scale |
| F4 | [1.50, 1.67, 1.63, 1.61] | Broad smooth landscape — EI safe with ±0.05 |
| F5 | [0.97, 8.39, 0.90, 13.8] | Dims 1&3 sensitive at boundary, dims 2&4 irrelevant |
| F6 | [11.6, 0.025, 24.5, 0.001, 0.0003] | Dims 4&5 hyper-sensitive — LOCK at best values |
| F7 | [0.74, 0.25, 1.18, 0.54, 0.26, 0.36] | Dims 2&5 most sensitive — GP suggests dim2-0.02 improves |
| F8 | [2.14, 3.77, 1.87, 3.16, 10.5, 1e5, 2.65, 7.9e4] | Dims 6&8 irrelevant — fine-tune dims 1&3 |

### Strategy per function:

| F | Strategy | Rationale |
|---|----------|-----------|
| F1 | **Manual** [0.418, 0.410] | dim1+0.005 nudge. W5's dim2+0.005 gave +47%. Now explore dim1 direction |
| F2 | **Manual** [0.705, 0.125] | dim1+0.005. W5 went dim1-0.004 and scored worse. Try opposite direction |
| F3 | **Manual** [0.348, 0.667, 0.439] | dim2-0.005 (safest single-dim nudge within 1/3 of length scale) |
| F4 | **EI** (xi=0.003), ±0.05 | Proven strategy — 4 consecutive improvements |
| F5 | **Manual** [1, 1, 1, 0.999] | Stay at corner. Tiny dim4 test (ls=13.8, won't matter) |
| F6 | **Manual** — lock dims 4&5, dim2+0.005 | Dims 4&5 caused W5 disaster. Lock them. Nudge dim2 only |
| F7 | **Manual** dim2-0.02, dim6-0.02 | GP predicts 1.829 (vs best 1.784). Conservative double nudge |
| F8 | **UCB** (kappa=0.2), tight bounds | Nearly converged. Fine-tune dims 1&3, lock dims 6&8 |

In [ ]:
import warnings
import importlib
import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import fit_gp, propose_next_point

week6_recommendations = {}

def get_best_point(fid):
    """Get the best observed point for a function"""
    best_idx = np.argmax(outputs[fid])
    return inputs[fid][best_idx].copy()

# F1: Manual — dim1 nudge +0.005 (W5's dim2+0.005 gave +47%, now try dim1)
week6_recommendations[1] = np.array([0.418000, 0.410000])

# F2: Manual — dim1 nudge +0.005 (opposite direction from W5 miss)
# Best at [0.700073, 0.125362]. W5 went dim1-0.004 → scored 0.547 (worse)
week6_recommendations[2] = np.array([0.705000, 0.125362])

# F3: Manual — dim2-0.005 (safest nudge, GP predicts ~same as best)
best3 = get_best_point(3)  # [0.347863, 0.672420, 0.439172]
week6_recommendations[3] = np.array([best3[0], best3[1] - 0.005, best3[2]])

# F4: EI ±0.05 — proven strategy, 4 consecutive improvements
best4 = get_best_point(4)
bounds_f4 = np.array([[max(0, best4[d] - 0.05), min(1, best4[d] + 0.05)] for d in range(4)])
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    pt4, _ = propose_next_point(inputs[4], outputs[4], bounds_f4, acq_func='EI', xi=0.003, n_restarts=50)
week6_recommendations[4] = pt4

# F5: Manual — stay at corner, tiny dim4 variation (ls=13.8, irrelevant)
week6_recommendations[5] = np.array([1.000000, 1.000000, 1.000000, 0.999000])

# F6: Manual — LOCK dims 4&5 at best values, nudge dim2 only
best6 = get_best_point(6)  # [0.755469, 0.270580, 0.644099, 0.672228, 0.162862]
week6_recommendations[6] = np.array([
    best6[0],           # dim1 same (ls=11.6, irrelevant)
    best6[1] + 0.005,   # dim2 nudge +0.005 (ls=0.025)
    best6[2],           # dim3 same (ls=24.5, irrelevant)
    best6[3],           # dim4 LOCKED (ls=0.001)
    best6[4]            # dim5 LOCKED (ls=0.0003)
])

# F7: Manual — dim2-0.02 + dim6-0.02 (GP predicts 1.829 vs best 1.784)
best7 = get_best_point(7)  # [0.0, 0.342263, 0.707844, 0.246481, 0.405689, 0.778028]
week6_recommendations[7] = np.array([
    best7[0],            # dim1 same
    best7[1] - 0.02,     # dim2 -0.02 (ls=0.25, 0.08 length scales)
    best7[2],            # dim3 same
    best7[3],            # dim4 same
    best7[4],            # dim5 same
    best7[5] - 0.02      # dim6 -0.02 (ls=0.36, 0.06 length scales)
])

# F8: UCB tight — fine-tune dims 1&3, lock dims 6&8
best8 = get_best_point(8)
radii_f8 = [0.03, 0.05, 0.03, 0.05, 0.05, 0.005, 0.05, 0.005]
bounds_f8 = np.array([[max(0, best8[d] - radii_f8[d]), min(1, best8[d] + radii_f8[d])] for d in range(8)])
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    pt8, _ = propose_next_point(inputs[8], outputs[8], bounds_f8, acq_func='UCB', kappa=0.2, n_restarts=50)
week6_recommendations[8] = pt8

# Sanity check all recommendations
print("Week 6 Recommendations")
print("=" * 80)
for fid in range(1, 9):
    X, y = inputs[fid], outputs[fid]
    rec = week6_recommendations[fid]
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        gp = fit_gp(X, y)
    pred, pred_std = gp.predict(rec.reshape(1, -1), return_std=True)
    dists = np.linalg.norm(X - rec, axis=1)
    min_dist = np.min(dists)
    best = np.max(y)
    
    strategy = {
        1: "Manual dim1+0.005",
        2: "Manual dim1+0.005 (opposite W5)",
        3: "Manual dim2-0.005",
        4: "EI ±0.05",
        5: "Manual corner, dim4=0.999",
        6: "Manual lock d4&d5, d2+0.005",
        7: "Manual d2-0.02, d6-0.02",
        8: "UCB tight, tune d1&d3"
    }
    
    print(f"F{fid} ({X.shape[1]}D)  best={best:.4f}  pred={pred[0]:.4f}±{pred_std[0]:.4f}  dist={min_dist:.4f}  {strategy[fid]}")
    print(f"  point: {rec}")
print("=" * 80)

## 6. Submission Format

In [ ]:
# Submission format
print("=" * 70)
print("WEEK 6 SUBMISSION")
print("=" * 70)

for fid in range(1, 9):
    pt = week6_recommendations[fid]
    formatted = '-'.join(f'{x:.6f}' for x in pt)
    print(f"Function {fid}:\t{formatted}")